# Ablation Study
- do it with and without acceleration for different context lengths

In [1]:
## Load necessary library files ##

import sys
sys.path.append('..')
from source.utils import get_sequence, DatasetConverter
from source.model.model import Model

## Load standard files ##
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
from torch import from_numpy as tnsr
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import torch.nn.functional as F
from tqdm import tqdm
import pickle 

In [2]:
## select device ##
device = torch.device("cpu")

print("Using device:", device)

Using device: cpu


In [3]:
total_samples, n_community, n_members = 5000000, 2, 3
total_layers, head_layers, short_term_memory = 3, 3, 4

context_depths = [2, 4, 6, 8]
context_length = [7, 13, 19, 25]
vocab_size = n_community*n_members + 1


In [4]:
reps = 10
res = []
repititions = []
context = []
samples_seen = []

for rep in tqdm(range(reps)):
    for id, context_depth in enumerate(context_depths):
        model = Model(
            total_layers = total_layers,
            num_layers_prediction_head = head_layers,

            # ---- Layer sizes ----
            vocab_size = vocab_size,                  # layer 0 input dimension
            hidden_sizes = [100, 100, 100],    # H0, H1
            embedding_dim = 30,

            # ---- Learning rates per layer ----
            lr_layers = 1e-4,   

            # ---- Optimizer type (user can choose) ----
            optimizer_class = torch.optim.Adam,
            optimizer_kwargs = {
                "weight_decay": 1e-12
            },

            # ---- Sleep hyperparameters ----
            short_term_memory = short_term_memory,
            context_tag_buffer_size=100,
            # ---- Misc ----
            recon_threshold = 1e-3,
            device = device
        )

        data = get_sequence(total_samples, n_community, n_members, context_depth=context_depth, train_percent=1.0)
        dataset = DatasetConverter(data, short_term_memory=short_term_memory)
        loader = DataLoader(dataset, batch_size=1, shuffle=False)
        
        ii = 0 
        h_ = None
        correct_ring = np.zeros(1000)
        for x, y in loader:
            x = x.to(device).long()
            y = y.to(device).long()
            logits, loss, recon_loss, h_ = model.wake_step(x, y, h_)


            with torch.no_grad():
                ii += 1
                pred_tok = logits.argmax(dim=-1)
                correct_ring[ii % 1000] = (pred_tok[0] == y[0, 0]).item()
                
                if ii%1000 == 0:
                    acc = np.sum(correct_ring) / (1000 if ii >= 1000 else ii)
                    res.append(acc)
                    samples_seen.append(ii)
                    repititions.append(rep)
                    context.append(id)


            if ii%20000==0:
                model.sleep(total_steps=64)


df = pd.DataFrame()
df['reps'] = repititions
df['samples seen'] = samples_seen
df['context required'] = context
df['Accuracy'] = res

with open('../pickle_files/ablation_with_acceleration.pickle', 'wb') as f:
    pickle.dump(df, f)

100%|██████████| 10/10 [75:56:35<00:00, 27339.51s/it]  


In [ ]:
reps = 10
res = []
repititions = []
context = []
samples_seen = []

for rep in tqdm(range(reps)):
    for id, context_depth in enumerate(context_depths):
        model = Model(
            total_layers = total_layers,
            num_layers_prediction_head = head_layers,

            # ---- Layer sizes ----
            vocab_size = vocab_size,                  # layer 0 input dimension
            hidden_sizes = [100, 100, 100],    # H0, H1
            embedding_dim = 30,

            # ---- Learning rates per layer ----
            lr_layers = 1e-4,   

            # ---- Optimizer type (user can choose) ----
            optimizer_class = torch.optim.Adam,
            optimizer_kwargs = {
                "weight_decay": 1e-12
            },

            # ---- Sleep hyperparameters ----
            short_term_memory = short_term_memory,
            context_tag_buffer_size=100,
            # ---- Misc ----
            recon_threshold = 1e-3,
            accelerate = False,
            device = device
        )

        data = get_sequence(total_samples, n_community, n_members, context_depth=context_depth, train_percent=1.0)
        dataset = DatasetConverter(data, short_term_memory=short_term_memory)
        loader = DataLoader(dataset, batch_size=1, shuffle=False)
        
        ii = 0 
        h_ = None
        correct_ring = np.zeros(1000)
        for x, y in loader:
            x = x.to(device).long()
            y = y.to(device).long()
            logits, loss, recon_loss, h_ = model.wake_step(x, y, h_)


            with torch.no_grad():
                ii += 1
                pred_tok = logits.argmax(dim=-1)
                correct_ring[ii % 1000] = (pred_tok[0] == y[0, 0]).item()
                
                if ii%1000 == 0:
                    acc = np.sum(correct_ring) / (1000 if ii >= 1000 else ii)
                    res.append(acc)
                    samples_seen.append(ii)
                    repititions.append(rep)
                    context.append(id)


            if ii%20000==0:
                model.sleep(total_steps=16)


df = pd.DataFrame()
df['reps'] = repititions
df['samples seen'] = samples_seen
df['context required'] = context
df['Accuracy'] = res

with open('../pickle_files/ablation_without_acceleration.pickle', 'wb') as f:
    pickle.dump(df, f)

 60%|██████    | 6/10 [63:32:13<39:32:01, 35580.37s/it] 